In [ ]:
# ==========================================
# 1. Import Necessary Libraries
# ==========================================
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# Set seed for reproducibility
np.random.seed(42)

# ==========================================
# 2. Synthetic Dataset Generation
# ==========================================
# Creating a dummy dataset with missing values and a target variable
n_samples = 1000
data = {
    'feature_1': np.random.normal(loc=10, scale=2, size=n_samples),
    'feature_2': np.random.normal(loc=50, scale=10, size=n_samples),
    'feature_3': np.random.normal(loc=0, scale=1, size=n_samples),
    'target': np.random.choice([0, 1], size=n_samples, p=[0.7, 0.3])
}

df = pd.DataFrame(data)

# Introduce some missing values artificially
df.loc[df['feature_1'] > 12, 'feature_1'] = np.nan

# Introduce a Target Leakage Feature intentionally for demonstration
# (A feature that contains direct information about the target)
df['leaky_feature'] = df['target'] * 2.5 + np.random.normal(0, 0.1, n_samples)

print("Dataset Preview:")
print(df.head())

# ==========================================
# 3. INCORRECT PRACTICE (Data Leakage Introduced)
# ==========================================
print("\n--- Running INCORRECT Pipeline (With Data Leakage) ---")

# Bug 1: Preprocessing before Train-Test Split (Train-Test Contamination)
imputer = SimpleImputer(strategy='mean')
scaler = StandardScaler()

# Imputing and scaling entire dataset at once
X_raw = df.drop(columns=['target'])
y = df['target']

X_imputed = imputer.fit_transform(X_raw)
X_scaled = scaler.fit_transform(X_imputed)

# Splitting after transformations (Data Leakage Has Occurred!)
X_train_bad, X_test_bad, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Training model
model_bad = LogisticRegression()
model_bad.fit(X_train_bad, y_train)

# Evaluation
y_pred_bad = model_bad.predict(X_test_bad)
print(f"Incorrect Approach Accuracy: {accuracy_score(y_test, y_pred_bad):.4f}")
print("Note: This metric is overly optimistic due to target leakage & test set contamination.")

# ==========================================
# 4. CORRECT PRACTICE (Preventing Data Leakage)
# ==========================================
print("\n--- Running CORRECT Pipeline (Preventing Data Leakage) ---")

# Step 1: Remove Leaky Feature
X_clean = df.drop(columns=['target', 'leaky_feature'])
y_clean = df['target']

# Step 2: Split BEFORE any preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=42
)

# Step 3: Build Scikit-learn Pipeline
# Pipelines fit imputers/scalers ONLY on training data and transform test data
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression())
])

# Fit on Train data only
pipeline.fit(X_train, y_train)

# Predict on Unseen Test data
y_pred_correct = pipeline.predict(X_test)
print(f"Correct Approach Accuracy: {accuracy_score(y_test, y_pred_correct):.4f}")

# Cross-Validation without Leakage
cv_scores = cross_val_score(pipeline, X_clean, y_clean, cv=5)
print(f"5-Fold Cross-Validation Accuracy: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")